In [1]:
from tkinter import Tk
from tkinter import filedialog
from skimage.io import imread, imsave, imshow
from skimage.measure import regionprops, regionprops_table
import os
import numpy as np
import pandas as pd
import glob
# 0.9 was too high-- many pixels of each granule overlap background--trying 0.25


For ease, will assume, I have created a directory structure that just has the upscaled cell labels in a sub folder /cell, the original
cell channel image in a sub folder /orCell and the rna labels in a subfolder /rna and will but the tabular output in a subfolder called /tables. The overarching folder structure will be the same 0116, 0123, 1120, 1208, lowChighI etc.

INSTRUCTIONS: Create a folder that includes the following subfolders: /cell, /orCell, and /rna. These need to contain the
cell label file, the original cell image file (1-channel) and the rna label file. The output is put in the /table folder, which is created if needed.

The current version is designed to work only on one RNA channel at a time (now C1, but switch comment # to switch to C4)
NOTE, if you will be running many files, it will make sense to change the /rna folder so that it has all of the rna label files in it.

In [2]:
root = Tk()
root.withdraw()
strtPath = "/Users/nelsonlab/"
dirName = filedialog.askdirectory(initialdir=strtPath)
cellFold = dirName + "/cell/cLabs/upscaled"                                # this subfolder has the cell label files
orCellFold = dirName + "/cell/origSize"                            # this subfolder has the original cell image channel
rLabFold = dirName + "/rna/rLabs"                                 # this subfolder has the RNA label files
splitFold = rLabFold + "/split_labels"                       # this subfolder will contain the split RNA label files (in cells, not in cells)
if not os.path.exists(splitFold):  
   os.mkdir(splitFold)  
tabFold = dirName + "/table"
if not os.path.exists(tabFold):  
   os.mkdir(tabFold)  
rLabFiles = os.listdir(rLabFold)
cfiles = os.listdir(cellFold)
print("cellFold=",cellFold)
print(cfiles)

cellFold= /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/cell/cLabs/upscaled
['C1_040924_ScrGFP_B1_RH_D3_K9_cp_masks.tif', 'C1_040924_ScrGFP_B2_RH_D3_K9_cp_masks.tif', 'C1_040924_ScrGFP_B3_LH_D3_K9_cp_masks.tif', 'C1_040924_ScrGFP_B4_LH_D3_K9_cp_masks.tif']


OG method - set thresholds in advance for puncta filtering, output puncta counts per cell 

In [3]:
threshFrac = 0.8  # fraction of each RNA granule that must be contained within cell
minSize = 10       # threshold pixels for keeping RNA granule
verbose = False

for filename in cfiles:
   if filename.endswith("_cp_masks.tif"):
      print (filename)
      filename_rna = filename.replace("_cp_masks.tif","_lab.tif")
      cFilePath = os.path.join(cellFold,filename)
      cLabs = imread(cFilePath)
      print("Reading: ",cFilePath)
      ncells = np.shape(np.unique(cLabs))[0]
      if verbose: 
          print("Cell shapes=",cLabs.shape) 
          print("#cells=",ncells)
      for rLabfname in rLabFiles:
          if rLabfname.endswith(filename_rna[2:]):
              rLabPath = os.path.join(rLabFold,rLabfname)         
              rseg = imread(rLabPath)
              print("Reading: ", rLabPath)
              if verbose: 
                  print("Rna shapes=",rseg.shape)
                  numRNA = np.max(rseg)
                  print("#rna=", numRNA)
              lab,cnt = np.unique(rseg, return_counts=True)
              splitfnameIn = rLabfname.replace("lab","inCells")
              splitfnameOut = rLabfname.replace("lab","outCells")
              splitInPath = os.path.join(splitFold,splitfnameIn)
              splitOutPath = os.path.join(splitFold,splitfnameOut)
              labNumfname = rLabfname.replace("lab.tif","labNums.txt")
              labNumPath = os.path.join(tabFold,labNumfname)
              lab,cnt = np.unique(rseg, return_counts=True)
# get areas of rna labels and write to file
              #dfAreas = pd.DataFrame({'label':lab,'count':cnt})
              rnaTabFile = "rLabs_" + rLabfname
              rnaTabFile = rnaTabFile.replace(".tif",".csv")
              #rnaTabPath = os.path.join(tabFold,rnaTabFile)
              #dfAreas.to_csv(rnaTabPath)            # include index for now, to exclude use index=False
              if verbose: print("wrote dfAreas to ",rnaTabPath)
            # include index for now, to exclude use index=False
# now will loop through cells
              nLabsPix = np.zeros(((ncells-1),3), dtype=int)
              allKeepLong = np.array([])
              allKeepLab = [0] * (ncells-1)                     # apparently this is the "pythonic" was to preallocate a list
# first get rna objects that overlap background, we will exclude these
              a0 = rseg[cLabs==0]
              lab0, cnt0 = np.unique(a0, return_counts=True)
              if verbose: print("lab0=",lab0, " cnt0=",cnt0)
         
#         for ic in range (5):      # uncomment and comment next line to just print out stuff about first 5 cells.
              ix = -1
              for ic in np.unique(cLabs):
                  if ic==0: continue
                  ix=ix + 1
                  a = rseg[cLabs==ic]             # get the pixels in rseg, corresponding to the cell
                  lab,cnt = np.unique(a, return_counts=True)
                  bOver = np.isin(lab, lab0)         # boolean for labels that overlap with bkgrnd
                  labOver = lab[bOver]               # set of overlapping labels
                  nOver = lab[np.invert(bOver)]      # non-overlapping labels
                  ocnt = cnt[bOver]                  # pixel counts in cell for overlapping labels
                  bOver0 = np.isin(lab0, lab)        # Boolean for background labels that overlap cell
                  ocnt0 = cnt0[bOver0]               # pixels in bkgrnd
                  inCfrac = ocnt/(ocnt+ocnt0)        # ratio in cell
                  if verbose:
                      print("cell",ic)
                      print("pix in cell=",len(a)," labels=",lab," counts=",cnt)
                      print ("bkgnd pix=", ocnt0)
                      print("inCfrac=",inCfrac)
                      print("# nonoverlapping labels-",len(nOver),"ovelapping=",labOver)
                  incOver = labOver[inCfrac > threshFrac] #overlapping labels to include
                  bInc = (np.isin(lab, incOver) | np.isin(lab, nOver))
                                       # keep the inc overlaps and the nonoverlaps
                  keepLab = lab[bInc] 
                  keepCnt = cnt[bInc]
                  keepLab = keepLab[keepCnt > minSize]
                  keepCnt = keepCnt[keepCnt > minSize]
# note, zero should be in val0, so should not be uval
                  nLabsPix[ix,0] = ic
                  nLabsPix[ix,1] = len(keepLab)
                  nLabsPix[ix,2] = np.sum(keepCnt)
                  allKeepLab[ix] = keepLab             # list of arrays
                  allKeepLong = np.concatenate((allKeepLong,keepLab), axis=0)
              dfIsect = pd.DataFrame(data = nLabsPix, index = range(1,(ncells)), columns = ["cellID","nLabs","nPix"])
              isectTabFile = rnaTabFile.replace("rLabs_","isects_")
              isectTabPath = os.path.join(tabFold,isectTabFile)
              dfIsect.to_csv(isectTabPath)
              f = open(labNumPath,'w')
              for cell in allKeepLab:
                  s = ",".join(str(x) for x in cell)
                  f.write(s+"\n")
              allKeepLong = np.unique(allKeepLong)
              rzeros = np.zeros(shape = rseg.shape, dtype=np.int16)
              rseg = np.array(rseg, dtype=np.int16)
              rIn = np.where(np.isin(rseg,allKeepLong),rseg,rzeros)
              rOut = np.where(np.isin(rseg,allKeepLong, assume_unique=False, invert=True),rseg, rzeros)
              imsave(splitInPath, rIn, check_contrast=False)
              imsave(splitOutPath, rOut, check_contrast=False)

C1_040924_ScrGFP_B1_RH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/cell/cLabs/upscaled/C1_040924_ScrGFP_B1_RH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/rna/rLabs/rna1_040924_ScrGFP_B1_RH_D3_K9_lab.tif


KeyboardInterrupt: 

Alt. Method - no pre-filtering required, outputs tables of all cell/RNA intersections w/ pixel counts 

In [4]:
import re
verbose = False

#rTabFiles = os.listdir(tabFold)
#intersections = [rf for rf in rTabFiles if rf.endswith("intersections.csv")]
#res = [fi.replace('intersections.csv', 'cp_masks.tif') for fi in intersections]
#res2 = [re.sub('rna[0-9]','C1',fi) for fi in res] 

for filename in cfiles:
    if filename.endswith("_cp_masks.tif"):
        #comment out if you don't want to reprocess files you've already done 
        #if filename in res2: continue
        print (filename)
        filename_rna = filename.replace("_cp_masks.tif","_lab.tif")
        cFilePath = os.path.join(cellFold,filename)
        cLabs = imread(cFilePath)
        print("Reading: ",cFilePath)
        ncells = np.shape(np.unique(cLabs))[0]
        if verbose:
            print("Cell shapes=",cLabs.shape) 
            print("#cells=",ncells)
        for rLabfname in rLabFiles:
            if rLabfname.endswith(filename_rna[2:]):
                rLabPath = os.path.join(rLabFold,rLabfname)         
                rseg = imread(rLabPath)
                print("Reading: ", rLabPath)
                numRNA = np.max(rseg)
                lab,cnt = np.unique(rseg, return_counts=True)
                dfIsect = pd.DataFrame(index = range(0), columns = ["cellID","labelID","nPixinCell","nPixinback","nPix","inCfrac"])

# first get rna objects that overlap background, we will exclude these
                a0 = rseg[cLabs==0]
                lab0, cnt0 = np.unique(a0, return_counts=True)
        # objects that do not overlap background
                lab01 = np.setdiff1d(range(numRNA + 1),lab0)
                cnt01 = [0] * np.size(lab01)
        #combine (sorted to reflect original indexes)
                labx = np.concatenate((lab0,lab01))
                idx1 = np.argsort(labx)
                lab0 = labx[idx1]
                cnt0 = np.concatenate((cnt0,cnt01))[idx1]
                ix = -1
                for ic in np.unique(cLabs):
                    if ic==0: continue
                    ix=ix + 1
                    a = rseg[cLabs==ic]             # get the pixels in rseg, corresponding to the cell
                    lab1,cnt1 = np.unique(a, return_counts=True)
                    ocnt0 = cnt0[lab1]
                    ocnt = cnt[lab1]
                    inCfrac = cnt1/ocnt        # ratio in cell
                                       # keep the inc overlaps and the nonoverlaps
                    icx = np.repeat(ic,np.size(lab1))
# note, zero should be in val0, so should not be uval
                    if np.size(lab1) > 0:
                        nLabsPix = np.column_stack((icx,lab1,cnt1,ocnt0,ocnt,inCfrac))
                        tempdf = pd.DataFrame(data = nLabsPix, index = range(np.size(lab1)), columns = ["cellID","labelID","nPixinCell","nPixinback","nPix","inCfrac"])
                        tempdf = tempdf[tempdf['labelID']!=0]
                        dfIsect = pd.concat((dfIsect,tempdf))
                isectTabFile = rLabfname.replace("lab.tif","intersections.csv")
                isectTabPath = os.path.join(tabFold,isectTabFile)
                dfIsect.to_csv(isectTabPath)

C1_040924_ScrGFP_B1_RH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/cell/cLabs/upscaled/C1_040924_ScrGFP_B1_RH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/rna/rLabs/rna1_040924_ScrGFP_B1_RH_D3_K9_lab.tif


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_83503/2380680385.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dfIsect = pd.concat((dfIsect,tempdf))


C1_040924_ScrGFP_B2_RH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/cell/cLabs/upscaled/C1_040924_ScrGFP_B2_RH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/rna/rLabs/rna1_040924_ScrGFP_B2_RH_D3_K9_lab.tif


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_83503/2380680385.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dfIsect = pd.concat((dfIsect,tempdf))


C1_040924_ScrGFP_B3_LH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/cell/cLabs/upscaled/C1_040924_ScrGFP_B3_LH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/rna/rLabs/rna1_040924_ScrGFP_B3_LH_D3_K9_lab.tif


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_83503/2380680385.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dfIsect = pd.concat((dfIsect,tempdf))


C1_040924_ScrGFP_B4_LH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/cell/cLabs/upscaled/C1_040924_ScrGFP_B4_LH_D3_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/P20_RNAScope/WorkingFile/rna/rLabs/rna1_040924_ScrGFP_B4_LH_D3_K9_lab.tif


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_83503/2380680385.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dfIsect = pd.concat((dfIsect,tempdf))
